# 05 · Fine-tuning de A-ESRGAN (brazos E/F/G)

**Qué hace:** entrena tres brazos de A-ESRGAN con el protocolo factorial de las fases 0/A/C y el módulo de degradación espectral del notebook 04 — E (270 imgs, espectral), F (740, espectral), G (740, genérica). F es el modelo del TFM. Fija particiones, construye los YAML como diferencia mínima sobre el brazo A, materializa los datos, lanza BasicSR, selecciona checkpoint y sintetiza.

**Qué necesita:**
- `artifacts/particiones.json`, `artifacts/meta_info_{E,F,G}.txt`, `configs/brazoA_nativo.yml`, `configs/brazoE.yml`/`brazoF.yml`/`brazoG.yml`, `configs/parametros_degradacion.json`, `dataset_espectral.py` (todo en el repo)
- Kaggle: `joe1995/div2k-dataset`, `daehoyang/flickr2k`
- GPU (Google Colab)
- **No incluido:** checkpoints y logs del brazo A (cadena Fase 0–3). Sin ellos, la comparación con A se omite; el entrenamiento de E/F/G funciona igual.

**Qué deja escrito:** checkpoints y métricas en `ROOT/_out/05/` (usa `montar_drive_opcional()` para persistir en Drive)

**Arranque:** primera celda de código.

## Configuración

In [ ]:
!pip -q install pyyaml lpips scikit-image pandas "basicsr>=1.3.3.11"
from colab_setup import aplicar_parches, preparar_repo, descargar_datos, montar_drive_opcional
aplicar_parches()
ROOT = preparar_repo()
DATA = descargar_datos("kaggle:div2k", "kaggle:flickr2k", root=ROOT)
CONFIGS   = ROOT / "configs"
ARTIFACTS = ROOT / "artifacts"
SALIDA    = ROOT / "_out" / "05"; SALIDA.mkdir(parents=True, exist_ok=True)

In [ ]:
import json, shutil
from pathlib import Path

import numpy as np
import pandas as pd
import cv2

# --- Manifiesto de particiones 0/A/C (vendorizado) ---
PARTICIONES_ORIG = ARTIFACTS / "particiones.json"

# --- DIV2K: carpeta plana con las 800 PNG. kagglehub puede anidarlas en un
#     subdirectorio, asi que se resuelve al directorio que las contiene. ---
_div2k_pngs = sorted(DATA["kaggle:div2k"].rglob("*.png"))
DIV2K    = _div2k_pngs[0].parent if _div2k_pngs else DATA["kaggle:div2k"]
FLICKR2K = DATA["kaggle:flickr2k"]

# --- Degradacion espectral del notebook 04. RealESRGANDatasetEspectral exige
#     psd_objetivo.npz JUNTO al JSON de parametros; se materializan ambos. ---
DEGRAD = SALIDA / "degradacion"; DEGRAD.mkdir(parents=True, exist_ok=True)
shutil.copy(CONFIGS / "parametros_degradacion.json", DEGRAD / "parametros_degradacion.json")
shutil.copy(ARTIFACTS / "psd_objetivo.npz",          DEGRAD / "psd_objetivo.npz")
PARAMS_ESPECTRAL = DEGRAD / "parametros_degradacion.json"

# --- Directorios de salida (efimeros en Colab; usar montar_drive_opcional() para persistir) ---
BRAZO_A_YAML       = CONFIGS / "brazoA_nativo.yml"
SALIDA_PARTICIONES = SALIDA / "particiones"
SALIDA_YAML        = SALIDA / "configs"
DATOS_DIR          = SALIDA / "datos"
DATOS_LOCAL        = Path("/content/datos_hr_05")     # symlinks locales; se pierden al reiniciar
CKPT_DIR           = SALIDA / "checkpoints"
RESULTADOS_DIR     = SALIDA / "resultados"
LOGS_DIR           = RESULTADOS_DIR
AESRGAN_DIR        = Path("/content/A-ESRGAN")
for _d in (SALIDA_PARTICIONES, SALIDA_YAML, DATOS_DIR, CKPT_DIR, RESULTADOS_DIR):
    _d.mkdir(parents=True, exist_ok=True)

# --- Cadena Fase 0-3 (NO versionada). Deja aqui sus salidas para reactivar la
#     comparacion con el brazo A: ROOT/_fases_0a3/Fase1/pares/test/{lr,hr},
#     ROOT/_fases_0a3/Fase2/checkpoints/brazoA_nativo/models, .../Fase2/sonda/... ---
FASES_0A3 = ROOT / "_fases_0a3"

assert PARTICIONES_ORIG.exists(), f"Falta {PARTICIONES_ORIG} (viene en el repo)"
assert PARAMS_ESPECTRAL.exists(), f"Falta {PARAMS_ESPECTRAL}"
print("DIV2K :", DIV2K, f"({len(_div2k_pngs)} PNG)")
print("Salida:", SALIDA)

## 1. Particiones y verificación

### Por qué el conjunto ampliado no puede ser "DIV2K completo" sin más

Los brazos 0, A y C se entrenaron sobre un corpus de 300 imágenes (semilla 1234): 270 de
entrenamiento, 30 de validación y 30 de test. Ese test —treinta imágenes con numeración
0310 a 0788— cae dentro del rango completo de DIV2K (0001–0800). Si F y G ampliasen el
entrenamiento a las 800 imágenes sin excluir nada, entrenarían sobre las mismas fotografías
que luego se usan para medir LPIPS y para el contraste de Wilcoxon: el resultado en test
quedaría inflado por memorización y la comparación factorial, que es el motivo de esta fase,
dejaría de ser válida.

El conjunto ampliado correcto son por tanto **740 imágenes**: las 800 de DIV2K menos las 30
de validación y las 30 de test, que se mantienen fijas e idénticas a las de 0/A/C en los
cinco brazos. Conviene dejar esta precisión en la memoria si en algún borrador anterior se
escribió "800".

In [ ]:
orig = json.loads(PARTICIONES_ORIG.read_text())
TRAIN_A, VAL, TEST = set(orig['train']), set(orig['val']), set(orig['test'])
assert len(TRAIN_A) == 270, f'se esperaban 270 en train de 0/A/C, hay {len(TRAIN_A)}'
assert len(VAL) == 30, f'se esperaban 30 en val, hay {len(VAL)}'
assert len(TEST) == 30, f'se esperaban 30 en test, hay {len(TEST)}'
print(f'particiones.json cargado: train={len(TRAIN_A)} val={len(VAL)} test={len(TEST)} '
      f'(semilla={orig["semilla"]}, n_corpus={orig["n_corpus"]})')

todos = {p.name for p in DIV2K.glob('*.png')}
print(f'DIV2K en disco: {len(todos)} imágenes')

faltan = (TRAIN_A | VAL | TEST) - todos
assert not faltan, f'Imágenes de particiones.json que no están en DIV2K: {sorted(faltan)[:5]}...'

excluidas = VAL | TEST
TRAIN_FG = sorted(todos - excluidas)
print(f'Train ampliado (F/G): {len(TRAIN_FG)} imágenes')
assert len(TRAIN_FG) == 740, (
    f'Se esperaban 740 (800 − 30 val − 30 test), hay {len(TRAIN_FG)}. '
    f'Revisar si DIV2K en disco tiene realmente 800 imágenes.'
)

In [ ]:
PARTICIONES = {
    'E': {'train': sorted(TRAIN_A), 'val': sorted(VAL), 'test': sorted(TEST), 'degradacion': 'espectral'},
    'F': {'train': TRAIN_FG,        'val': sorted(VAL), 'test': sorted(TEST), 'degradacion': 'espectral'},
    'G': {'train': TRAIN_FG,        'val': sorted(VAL), 'test': sorted(TEST), 'degradacion': 'generica'},
}

filas = []
for brazo, p in PARTICIONES.items():
    tr, va, te = set(p['train']), set(p['val']), set(p['test'])
    filas.append({
        'brazo': brazo,
        'train': len(tr), 'val': len(va), 'test': len(te),
        'solape_train_val': len(tr & va),
        'solape_train_test': len(tr & te),
        'solape_val_test': len(va & te),
        'test_igual_0AC': te == TEST,
        'val_igual_0AC': va == VAL,
    })
TABLA_PARTICIONES = pd.DataFrame(filas)
display(TABLA_PARTICIONES)

sin_solape = (TABLA_PARTICIONES[['solape_train_val', 'solape_train_test', 'solape_val_test']] == 0).all().all()
sets_iguales = TABLA_PARTICIONES['test_igual_0AC'].all() and TABLA_PARTICIONES['val_igual_0AC'].all()
assert sin_solape, 'Hay solape entre particiones dentro de algún brazo'
assert sets_iguales, 'El val o el test de algún brazo no coincide exactamente con 0/A/C'
print(f'\nSin solape interno: {sin_solape}')
print(f'val/test idénticos a 0/A/C en los tres brazos: {sets_iguales}')

In [ ]:
for brazo, p in PARTICIONES.items():
    (SALIDA_PARTICIONES/f'particiones_{brazo}.json').write_text(json.dumps(p, indent=2))
print('Manifiestos escritos en', SALIDA_PARTICIONES)
for f in sorted(SALIDA_PARTICIONES.glob('*.json')):
    print(' ', f.name)

## 2. Métricas por imagen (CIEDE2000 validado)


Sharma, Wu y Dalal (2005) publicaron una tabla de 29 pares L\*a\*b\* con el ΔE00 esperado,
que se ha convertido en el test de facto para cualquier implementación de la fórmula: es
notoriamente fácil de programar mal por errores de signo o de orden en los términos
angulares, y el error no se manifiesta como un fallo sino como una cifra ligeramente
incorrecta que nadie nota. Validar contra ella es más fiable que revisar la fórmula a ojo.

In [ ]:
def delta_e00(lab1, lab2):
    '''CIEDE2000. lab1, lab2: arrays (...,3) con L*,a*,b* en la convención CIE estándar.'''
    L1, a1, b1 = lab1[..., 0], lab1[..., 1], lab1[..., 2]
    L2, a2, b2 = lab2[..., 0], lab2[..., 1], lab2[..., 2]

    Cb1 = np.sqrt(a1**2 + b1**2)
    Cb2 = np.sqrt(a2**2 + b2**2)
    Cb_mean = (Cb1 + Cb2) / 2.0
    G = 0.5 * (1 - np.sqrt(Cb_mean**7 / (Cb_mean**7 + 25.0**7 + 1e-30)))
    a1p, a2p = (1 + G) * a1, (1 + G) * a2

    C1p = np.sqrt(a1p**2 + b1**2)
    C2p = np.sqrt(a2p**2 + b2**2)

    def hp_of(ap, b):
        h = np.degrees(np.arctan2(b, ap))
        return np.where(h < 0, h + 360, h)
    h1p = np.where((a1p == 0) & (b1 == 0), 0.0, hp_of(a1p, b1))
    h2p = np.where((a2p == 0) & (b2 == 0), 0.0, hp_of(a2p, b2))

    dLp = L2 - L1
    dCp = C2p - C1p

    dhp = h2p - h1p
    dhp = np.where(C1p * C2p == 0, 0.0, dhp)
    dhp = np.where(dhp > 180, dhp - 360, dhp)
    dhp = np.where(dhp < -180, dhp + 360, dhp)
    dHp = 2 * np.sqrt(C1p * C2p) * np.sin(np.radians(dhp / 2.0))

    Lp_mean = (L1 + L2) / 2.0
    Cp_mean = (C1p + C2p) / 2.0

    hp_sum = h1p + h2p
    hp_mean = np.where(
        C1p * C2p == 0, hp_sum,
        np.where(np.abs(h1p - h2p) <= 180, hp_sum / 2.0,
                np.where(hp_sum < 360, (hp_sum + 360) / 2.0, (hp_sum - 360) / 2.0))
    )

    T = (1 - 0.17 * np.cos(np.radians(hp_mean - 30))
           + 0.24 * np.cos(np.radians(2 * hp_mean))
           + 0.32 * np.cos(np.radians(3 * hp_mean + 6))
           - 0.20 * np.cos(np.radians(4 * hp_mean - 63)))

    d_theta = 30 * np.exp(-(((hp_mean - 275) / 25.0) ** 2))
    Rc = 2 * np.sqrt(Cp_mean**7 / (Cp_mean**7 + 25.0**7 + 1e-30))
    Sl = 1 + (0.015 * (Lp_mean - 50) ** 2) / np.sqrt(20 + (Lp_mean - 50) ** 2)
    Sc = 1 + 0.045 * Cp_mean
    Sh = 1 + 0.015 * Cp_mean * T
    Rt = -np.sin(np.radians(2 * d_theta)) * Rc

    return np.sqrt(
        (dLp / Sl) ** 2 + (dCp / Sc) ** 2 + (dHp / Sh) ** 2 + Rt * (dCp / Sc) * (dHp / Sh)
    )

In [ ]:
# Tabla de referencia de Sharma, Wu y Dalal (2005): L1,a1,b1, L2,a2,b2, dE00_esperado
CASOS_REFERENCIA = [
    (50.0000, 2.6772, -79.7751, 50.0000, 0.0000, -82.7485, 2.0425),
    (50.0000, 3.1571, -77.2803, 50.0000, 0.0000, -82.7485, 2.8615),
    (50.0000, 2.8361, -74.0200, 50.0000, 0.0000, -82.7485, 3.4412),
    (50.0000, -1.3802, -84.2814, 50.0000, 0.0000, -82.7485, 1.0000),
    (50.0000, -1.1848, -84.8006, 50.0000, 0.0000, -82.7485, 1.0000),
    (50.0000, -0.9009, -85.5211, 50.0000, 0.0000, -82.7485, 1.0000),
    (50.0000, 0.0000, 0.0000, 50.0000, -1.0000, 2.0000, 2.3669),
    (50.0000, -1.0000, 2.0000, 50.0000, 0.0000, 0.0000, 2.3669),
    (50.0000, 2.4900, -0.0010, 50.0000, -2.4900, 0.0009, 7.1792),
    (50.0000, 2.4900, -0.0010, 50.0000, -2.4900, 0.0010, 7.1792),
    (50.0000, 2.4900, -0.0010, 50.0000, -2.4900, 0.0011, 7.2195),
    (50.0000, 2.5000, 0.0000, 73.0000, 25.0000, -18.0000, 27.1492),
    (50.0000, 2.5000, 0.0000, 61.0000, -5.0000, 29.0000, 22.8977),
    (50.0000, 2.5000, 0.0000, 56.0000, -27.0000, -3.0000, 31.9030),
    (50.0000, 2.5000, 0.0000, 58.0000, 24.0000, 15.0000, 19.4535),
    (50.0000, 2.5000, 0.0000, 50.0000, 3.1736, 0.5854, 1.0000),
    (50.0000, 2.5000, 0.0000, 50.0000, 3.2972, 0.0000, 1.0000),
    (50.0000, 2.5000, 0.0000, 50.0000, 1.8634, 0.5757, 1.0000),
    (50.0000, 2.5000, 0.0000, 50.0000, 3.2592, 0.3350, 1.0000),
    (60.2574, -34.0099, 36.2677, 60.4626, -34.1751, 39.4387, 1.2644),
    (63.0109, -31.0961, -5.8663, 62.8187, -29.7946, -4.0864, 1.2630),
    (61.2901, 3.7196, -5.3901, 61.4292, 2.2480, -4.9620, 1.8731),
    (35.0831, -44.1164, 3.7933, 35.0232, -40.0716, 1.5901, 1.8645),
    (22.7233, 20.0904, -46.6940, 23.0331, 14.9730, -42.5619, 2.0373),
    (36.4612, 47.8580, 18.3852, 36.2715, 50.5065, 21.2231, 1.4146),
    (90.8027, -2.0831, 1.4410, 91.1528, -1.6435, 0.0447, 1.4441),
    (90.9257, -0.5406, -0.9208, 88.6381, -0.8985, -0.7239, 1.5381),
    (6.7747, -0.2908, -2.4247, 5.8714, -0.0985, -2.2286, 0.6377),
    (2.0776, 0.0795, -1.1350, 0.9033, -0.0636, -0.5514, 0.9082),
]

errores = []
for L1, a1, b1, L2, a2, b2, esperado in CASOS_REFERENCIA:
    medido = float(delta_e00(np.array([L1, a1, b1]), np.array([L2, a2, b2])))
    errores.append(abs(medido - esperado))

print(f'error máximo frente a la tabla de referencia: {max(errores):.5f}  (tolerancia 0,01)')
assert max(errores) < 0.01, 'La implementación de CIEDE2000 no reproduce la tabla de referencia'
print(f'CIEDE2000 validado contra los {len(CASOS_REFERENCIA)} casos de Sharma et al. (2005).')

### Métricas por imagen

Una función por par de imágenes, con todo lo necesario para el contraste de Wilcoxon
posterior: PSNR, SSIM, MAE, RMSE, ΔE00 y C\*. Se apoya en la función de ΔE00 ya validada, y
convierte el espacio de color de OpenCV —que escala L a [0, 255] y centra a/b en 128— a la
convención CIE estándar antes de calcular nada, porque aplicar CIEDE2000 directamente sobre
la salida de OpenCV sin reescalar es un error habitual y silencioso.

In [ ]:
from skimage.metrics import peak_signal_noise_ratio as _psnr, structural_similarity as _ssim

def metricas_par(restaurada_bgr, referencia_bgr):
    '''Métricas de una imagen restaurada frente a su referencia. Ambas uint8 BGR, mismo tamaño.'''
    assert restaurada_bgr.shape == referencia_bgr.shape, 'restaurada y referencia deben tener el mismo tamaño'
    r = restaurada_bgr.astype(np.float64) / 255.
    g = referencia_bgr.astype(np.float64) / 255.

    mae = float(np.abs(r - g).mean())
    rmse = float(np.sqrt(((r - g) ** 2).mean()))
    with np.errstate(divide='ignore'):
        psnr = float(_psnr(g, r, data_range=1.0))   # inf si las imágenes son idénticas: es correcto, no un error
    ssim = float(_ssim(g, r, data_range=1.0, channel_axis=2))

    lab_r = cv2.cvtColor(restaurada_bgr, cv2.COLOR_BGR2LAB).astype(np.float64)
    lab_g = cv2.cvtColor(referencia_bgr, cv2.COLOR_BGR2LAB).astype(np.float64)
    lab_r[..., 0] *= 100. / 255.;  lab_g[..., 0] *= 100. / 255.
    lab_r[..., 1:] -= 128.;        lab_g[..., 1:] -= 128.

    de00 = float(np.mean(delta_e00(lab_r, lab_g)))
    c_estrella = float(np.mean(np.sqrt(lab_r[..., 1] ** 2 + lab_r[..., 2] ** 2)))
    return dict(mae=mae, rmse=rmse, psnr=psnr, ssim=ssim, delta_e00=de00, c_estrella=c_estrella)

### Comprobaciones de sanidad

Cinco casos que cualquier función de métricas debe superar antes de confiarse: dar valores
nulos/máximos frente a sí misma, coincidir con la fórmula cerrada del PSNR, empeorar de
forma monótona con más ruido, reaccionar a un viraje de color puro, y fallar de forma
explícita —no en silencio— si las dos imágenes no miden lo mismo.

In [ ]:
rng = np.random.default_rng(0)
base = rng.integers(40, 215, (128, 128, 3)).astype(np.uint8)

# Caso 1: identica a si misma
m = metricas_par(base, base)
assert m['mae'] == 0 and m['rmse'] == 0 and m['ssim'] > 0.999 and m['delta_e00'] < 1e-6
print('OK  imagen idéntica: métricas nulas/máximas según corresponde')

# Caso 2: PSNR coincide con la formula cerrada
ruidosa = np.clip(base.astype(np.float64) + rng.normal(0, 15, base.shape), 0, 255).astype(np.uint8)
mse = float((((ruidosa.astype(np.float64) - base.astype(np.float64)) / 255.) ** 2).mean())
psnr_manual = 10 * np.log10(1.0 / mse)
assert abs(metricas_par(ruidosa, base)['psnr'] - psnr_manual) < 0.01
print('OK  PSNR coincide con la fórmula cerrada')

# Caso 3: monotonia frente al ruido
serie = [metricas_par(np.clip(base.astype(np.float64) + rng.normal(0, s, base.shape), 0, 255).astype(np.uint8), base)
         for s in (5, 15, 30, 60)]
assert all(serie[i]['psnr'] > serie[i+1]['psnr'] for i in range(3))
assert all(serie[i]['ssim'] > serie[i+1]['ssim'] for i in range(3))
print('OK  PSNR y SSIM empeoran monótonamente con más ruido')

# Caso 4: viraje de color puro
sepia = np.clip(base.astype(np.float64) * [0.9, 1.0, 1.3], 0, 255).astype(np.uint8)
assert metricas_par(sepia, base)['delta_e00'] > 2.0
print('OK  ΔE00 reacciona a un viraje de color visible')

# Caso 5: tamaños distintos fallan explicitamente
try:
    metricas_par(base[:100, :100], base)
    raise RuntimeError('no debería haber pasado')
except AssertionError:
    print('OK  tamaños distintos lanzan un error explícito, no un resultado silencioso')

### Evaluación de un directorio completo

Recorre todas las imágenes restauradas de una carpeta, empareja cada una con su referencia
homónima y devuelve un DataFrame con una fila por imagen. Una imagen problemática —sin
referencia, de tamaño distinto, o ilegible— se marca y no interrumpe el resto: en una
evaluación de decenas de imágenes conviene saber cuál falló, no perder las demás porque una
esté mal.

In [ ]:
def evaluar_directorio(dir_restauradas, dir_referencia, calcular_lpips=False, lpips_fn=None):
    filas = []
    for f in sorted(Path(dir_restauradas).glob('*.png')):
        ref_path = Path(dir_referencia) / f.name
        fila = {'imagen': f.name, 'error': ''}
        if not ref_path.exists():
            fila['error'] = 'sin referencia'; filas.append(fila); continue
        r = cv2.imread(str(f), cv2.IMREAD_COLOR)
        g = cv2.imread(str(ref_path), cv2.IMREAD_COLOR)
        if r is None or g is None:
            fila['error'] = 'ilegible'; filas.append(fila); continue
        if r.shape != g.shape:
            fila['error'] = f'tamaños distintos {r.shape[:2]} vs {g.shape[:2]}'; filas.append(fila); continue
        m = metricas_par(r, g)
        m['error'] = ''          # ← garantiza que error esté presente tras el update
        fila.update(m)
        if calcular_lpips:
            fila['lpips'] = lpips_fn(r, g)
        filas.append(fila)
    return pd.DataFrame(filas)

### Prueba con incidencias provocadas a propósito

Cuatro imágenes de prueba: una válida, una sin referencia, una con tamaño distinto y una
ilegible. Las cuatro deben aparecer en el resultado, cada una con el error que le
corresponde, y ninguna debe tirar abajo el bucle.

In [ ]:
_t_rest, _t_ref = Path('/tmp/_t_rest'), Path('/tmp/_t_ref')
_t_rest.mkdir(exist_ok=True); _t_ref.mkdir(exist_ok=True)
for p in list(_t_rest.glob('*')) + list(_t_ref.glob('*')):
    p.unlink()

_base = rng.integers(40, 215, (64, 64, 3)).astype(np.uint8)
cv2.imwrite(str(_t_rest/'img_ok.png'), _base)
cv2.imwrite(str(_t_ref/'img_ok.png'), np.clip(_base.astype(int) + rng.integers(-5, 5, _base.shape), 0, 255).astype(np.uint8))
cv2.imwrite(str(_t_rest/'img_sin_ref.png'), _base)
cv2.imwrite(str(_t_rest/'img_tam.png'), _base)
cv2.imwrite(str(_t_ref/'img_tam.png'), _base[:50, :50])
open(_t_rest/'img_roto.png', 'wb').write(b'no es un png valido')
open(_t_ref/'img_roto.png', 'wb').write(b'tampoco esto')

_df_prueba = evaluar_directorio(_t_rest, _t_ref)
print(_df_prueba[['imagen', 'error']].to_string(index=False))

_esperado = {'img_ok.png': '', 'img_sin_ref.png': 'sin referencia',
            'img_tam.png': 'tamaños distintos (64, 64) vs (50, 50)', 'img_roto.png': 'ilegible'}
for _, fila in _df_prueba.iterrows():
    assert fila['error'] == _esperado[fila['imagen']], f"fallo en {fila['imagen']}"
assert len(_df_prueba) == 4, 'no deberían perderse filas'
print('\nOK: las cuatro incidencias se clasifican bien y ninguna interrumpe el bucle')

### LPIPS

Esta pieza **no se ha podido ejecutar** en el entorno donde se preparó este notebook, porque
no dispone de GPU ni de PyTorch. La función sigue la interfaz estándar del paquete `lpips`
—modelo AlexNet, entrada normalizada a [-1, 1]— y el modelo se carga una sola vez por sesión,
no en cada llamada, porque cargarlo por imagen sería prohibitivo en una evaluación de
decenas de imágenes por punto de control. Conviene ejecutar la celda siguiente y comprobar
que el valor de la imagen idéntica sale en 0,0 antes de fiarse del resto.

In [ ]:
import torch, lpips

_LPIPS_MODELO = None
def lpips_par(restaurada_bgr, referencia_bgr):
    global _LPIPS_MODELO
    if _LPIPS_MODELO is None:
        _LPIPS_MODELO = lpips.LPIPS(net='alex')
        if torch.cuda.is_available():
            _LPIPS_MODELO = _LPIPS_MODELO.cuda()
    def _a_tensor(img_bgr):
        rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 127.5 - 1.0
        t = torch.from_numpy(rgb.transpose(2, 0, 1)).unsqueeze(0)
        return t.cuda() if torch.cuda.is_available() else t
    with torch.no_grad():
        d = _LPIPS_MODELO(_a_tensor(restaurada_bgr), _a_tensor(referencia_bgr))
    return float(d.item())

# Verificación mínima: LPIPS de una imagen consigo misma debe ser (casi) 0
_d_ident = lpips_par(_base, _base)
print(f'LPIPS imagen idéntica: {_d_ident:.6f}  (debe ser ~0)')
assert _d_ident < 1e-3, 'LPIPS no da ~0 para la imagen idéntica: revisar antes de usarlo en el entrenamiento'
print('OK: LPIPS pasa la comprobación mínima. Verificado en Colab, no en el entorno de preparación.')

## 3. YAML de los brazos (diferencia mínima sobre A)


El YAML real del brazo A aclara dos cosas que no se podían dar por supuestas. Primera: A
entrena degradando en GPU a partir de HR sin degradar, mediante `RealESRGANDataset`, y no hay
ningún bloque `datasets.val`, así que la selección de punto de control se hace fuera de
BasicSR evaluando los checkpoints guardados cada 100 iteraciones con las funciones de la sección
2. Segunda: el tamaño de parche de entrenamiento es 128, distinto del 1024/256 que se usó en
la Fase 4b para *medir* el espectro de la degradación —son parámetros con el mismo nombre y
propósitos distintos.

Los tres brazos se construyen por diferencia mínima contra ese YAML: E y F cambian el
conjunto de datos y desactivan la degradación en GPU porque reciben el par ya degradado; G
cambia únicamente el conjunto de datos. Ningún hiperparámetro se reescribe de memoria.

In [ ]:
import yaml, copy, sys, inspect

### El YAML de A y el registro de BasicSR

El repositorio de A-ESRGAN no se instala por pip: se clona y se importa con `sys.path`
apuntando a su raíz, tal como hace la Fase 2. El paquete se llama `aesrgan`, y las clases se
registran al importarlo.

In [ ]:
assert BRAZO_A_YAML.exists(), f'Falta el YAML real de A en {BRAZO_A_YAML}'
A_YAML = yaml.safe_load(BRAZO_A_YAML.read_text())
print('YAML de A cargado:', BRAZO_A_YAML.name)
print('model_type declarado:', A_YAML['model_type'], '| dataset:', A_YAML['datasets']['train']['type'])
print('gt_size:', A_YAML['gt_size'], '| total_iter:', A_YAML['train']['total_iter'],
      '| checkpoint cada:', A_YAML['logger']['save_checkpoint_freq'])

In [ ]:
import subprocess

AESRGAN_DIR = Path('/content/A-ESRGAN')
PESOS_PRE   = AESRGAN_DIR / 'experiments' / 'pretrained_models' / 'A_ESRGAN_Single.pth'

REPO_URL    = 'https://github.com/stroking-fishes-ml-corp/A-ESRGAN.git'
MODEL_URL   = ('https://github.com/stroking-fishes-ml-corp/A-ESRGAN/releases/download/v1.0.0/A_ESRGAN_Single.pth')
if not AESRGAN_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(AESRGAN_DIR)], check=True)
if not PESOS_PRE.exists():
    PESOS_PRE.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['wget', '-q', '-O', str(PESOS_PRE), MODEL_URL], check=True)

In [ ]:
AESRGAN_DIR = Path('/content/A-ESRGAN')
assert AESRGAN_DIR.exists(), f'Clonar primero el repositorio en {AESRGAN_DIR} (ver Fase 2)'
if str(AESRGAN_DIR) not in sys.path:
    sys.path.insert(0, str(AESRGAN_DIR))

import aesrgan                              # registra las clases propias de A-ESRGAN
from basicsr.utils.registry import ARCH_REGISTRY, MODEL_REGISTRY, DATASET_REGISTRY

print('modelos registrados        :', sorted(MODEL_REGISTRY._obj_map))
print('discriminadores registrados:', [k for k in ARCH_REGISTRY._obj_map if 'iscrimin' in k])

### El nombre del modelo: la misma trampa que el discriminador

El YAML archivado del brazo A declara `model_type: RealESRGANModel`, pero ese nombre **no
está registrado**. BasicSR registra su modelo de Real-ESRGAN con `register(suffix='basicsr')`,
de modo que aparece como `RealESRGANModel_basicsr`; el nombre sin sufijo solo existiría si
además estuviera instalado el paquete `realesrgan`, que aquí no lo está. Es la misma trampa
que ya obligó a escribir `UNetDiscriminatorSN_basicsr` en `network_d`, solo que en la clave
`model_type` y sin que se hubiera detectado hasta ahora.

La alternativa disponible, `AESRGANModel`, es la clase propia del repositorio y usaría el
esquema adversario de A-ESRGAN. No procede aquí: la memoria documenta que el ajuste se hizo
con el esquema de Real-ESRGAN y con `UNetDiscriminatorSN`, sin el discriminador multiescala
con atención que constituye la aportación propia de A-ESRGAN.

In [ ]:
modelos = sorted(MODEL_REGISTRY._obj_map)
declarado = A_YAML['model_type']

if declarado in modelos:
    MODEL_TYPE = declarado
    print(f"'{declarado}' está registrado tal cual.")
elif f'{declarado}_basicsr' in modelos:
    MODEL_TYPE = f'{declarado}_basicsr'
    print(f"'{declarado}' NO está registrado; se usa '{MODEL_TYPE}' "
          f'(convención de sufijo de BasicSR).')
else:
    raise AssertionError(f"Ni '{declarado}' ni '{declarado}_basicsr' están registrados. "
                         f'Disponibles: {modelos}')

# El modelo elegido debe admitir la opción de la que dependen E y F
fuente = inspect.getsource(MODEL_REGISTRY.get(MODEL_TYPE).feed_data)
assert 'high_order_degradation' in fuente, (
    f'{MODEL_TYPE}.feed_data no consulta high_order_degradation: E y F no podrían recibir '
    'el par del conjunto de datos. Revisar antes de entrenar.'
)
print(f'OK  {MODEL_TYPE}.feed_data consulta high_order_degradation.')

assert A_YAML['network_d']['type'] in ARCH_REGISTRY._obj_map, (
    f"{A_YAML['network_d']['type']} no está registrado"
)
print(f"OK  discriminador '{A_YAML['network_d']['type']}' registrado.")

### Qué entrenó realmente el brazo A

Que el YAML archivado declare un `model_type` no registrado significa que **no es
literalmente el fichero que se ejecutó**, o que el entorno de entonces tenía instalado el
paquete `realesrgan`. La distinción importa: si A entrenó con un modelo distinto del que van
a usar E, F y G, la comparación factorial no sería válida.

BasicSR vuelca la configuración resuelta al arrancar, de modo que el registro de la Fase 2 es
la fuente autoritativa. La celda siguiente lo consulta si está disponible; si no lo encuentra,
avisa en lugar de dar el supuesto por bueno.

In [ ]:
LOG_A = FASES_0A3 / 'Fase2' / 'resultados' / 'log_brazoA_nativo.txt'
if LOG_A.exists():
    texto = LOG_A.read_text(errors='ignore')
    lineas = [ln.strip() for ln in texto.splitlines() if 'model_type' in ln]
    print(f'{LOG_A.name}: {len(texto.splitlines())} líneas')
    if lineas:
        print('model_type según el log:')
        for ln in lineas[:5]:
            print('  ', ln[:120])
    else:
        print('model_type no aparece explícitamente; se busca la línea de creación del modelo:')
        for ln in texto.splitlines():
            if 'Model [' in ln:
                print('  ', ln.strip()[:120]); break
else:
    print(f'[insumo externo ausente] {LOG_A} — el volcado de configuración resuelta '
          f'del brazo A pertenece a la cadena Fase 0–3, fuera de este repositorio. No se '
          f'confirma automáticamente con qué modelo entrenó A (ver sección siguiente).')

### Los tres YAML, como diferencia mínima sobre A

Las claves de degradación en GPU dejan de aplicarse en E y F, que reciben el par ya degradado
por `RealESRGANDatasetEspectral`: se retiran explícitamente en vez de dejarlas sin uso, para
que el YAML no sugiera un comportamiento que no tiene. G, en cambio, es literalmente A con
más imágenes.

**Cómo se desactiva la degradación en GPU.** El `feed_data` del modelo consulta
`high_order_degradation`, y cuando vale `false` toma directamente `lq` y `gt` del conjunto de
datos en lugar de sintetizarlos. Los tres brazos comparten así modelo, pérdidas y
discriminador con A: la única diferencia es de dónde sale el par degradado. Las claves
`*_gt_usm` se conservan, porque el realce sobre `gt` se sigue aplicando en esa rama.

In [ ]:
# Claves de degradacion en GPU que dejan de aplicarse con high_order_degradation: false
CLAVES_DEGRADACION_TOP = [
    'resize_prob', 'resize_range', 'gaussian_noise_prob', 'noise_range',
    'poisson_scale_range', 'gray_noise_prob', 'jpeg_range', 'second_blur_prob',
    'resize_prob2', 'resize_range2', 'gaussian_noise_prob2', 'noise_range2',
    'poisson_scale_range2', 'gray_noise_prob2', 'jpeg_range2', 'queue_size',
]

def _flatten(d, prefijo=''):
    out = {}
    for k, v in d.items():
        clave = f'{prefijo}.{k}' if prefijo else k
        out.update(_flatten(v, clave) if isinstance(v, dict) else {clave: v})
    return out

def informe_diff(base, nuevo, nombre):
    fb, fn = _flatten(base), _flatten(nuevo)
    cambios = [(k, fb.get(k, '<ausente>'), fn.get(k, '<ausente>'))
               for k in sorted(set(fb) | set(fn)) if fb.get(k, '<ausente>') != fn.get(k, '<ausente>')]
    print(f'--- {nombre}: {len(cambios)} claves distintas de A ---')
    for k, vb, vn in cambios:
        print(f'  {k:38s} {str(vb)[:26]:26s} -> {str(vn)[:36]}')
    return cambios


def construir_arm(letra, degradacion):
    y = copy.deepcopy(A_YAML)
    y['name'] = f'brazo{letra}'
    y['model_type'] = MODEL_TYPE          # nombre resuelto contra el registro real
    ruta_datos = DATOS_DIR

    if degradacion == 'generica':
        y['datasets']['train']['name'] = f'vintage_hr_{letra}'
        y['datasets']['train']['dataroot_gt'] = str(ruta_datos/f'hr_train_{letra}')
        y['datasets']['train']['meta_info'] = str(ruta_datos/f'meta_train_{letra}.txt')

    elif degradacion == 'espectral':
        for k in CLAVES_DEGRADACION_TOP:
            y.pop(k, None)
        y['high_order_degradation'] = False
        t = A_YAML['datasets']['train']
        y['datasets']['train'] = {
            'name': f'vintage_hr_{letra}', 'type': 'RealESRGANDatasetEspectral',
            # En la generación del YAML de cada brazo E/F/G:
            'dataroot_gt': str(DATOS_LOCAL / f'hr_train_{letra}'),
            # meta_info omitido: RealESRGANDatasetEspectral descubre imágenes con rglob.
            'ruta_parametros': str(PARAMS_ESPECTRAL),
            'io_backend': {'type': 'disk'},
            'gt_size': t['gt_size'],                      # igual que A: 128, no el de la calibración 4b
            'use_hflip': t['use_hflip'], 'use_rot': t['use_rot'], 'use_shuffle': t['use_shuffle'],
            'num_worker_per_gpu': t['num_worker_per_gpu'], 'batch_size_per_gpu': t['batch_size_per_gpu'],
            'dataset_enlarge_ratio': t['dataset_enlarge_ratio'], 'prefetch_mode': t['prefetch_mode'],
        }
    return y


DEGRADACIONES = {'E': 'espectral', 'F': 'espectral', 'G': 'espectral'}
YAMLS = {letra: construir_arm(letra, deg) for letra, deg in DEGRADACIONES.items()}

# G: pipeline espectral nulo — beta y sigma_blur fijos a 0 para no alterar la imagen.
# Equivalente a la degradación genérica en GPU pero con infraestructura robusta de carga.
YAMLS['G']['datasets']['train']['beta_rango']        = [0.0, 0.0]
YAMLS['G']['datasets']['train']['sigma_blur_rango']  = [0.0, 0.0]
YAMLS['G']['datasets']['train']['sigma_grano_rango'] = [0.0, 0.0]
YAMLS['G']['datasets']['train']['sigma_escaneo_rango'] = [0.0, 0.0]
YAMLS['G']['datasets']['train']['jpeg_rango']        = [100, 100]

for letra, y in YAMLS.items():
    informe_diff(A_YAML, y, f'brazo {letra}')
    print()

### Lo que no debe cambiar, verificado y no solo revisado a ojo

Un diff es fácil de leer mal cuando tiene cuarenta líneas. La comprobación siguiente falla
explícitamente si algo que el diseño factorial exige mantener fijo —arquitectura, optimizador,
pérdidas, semilla, tamaño de lote, tamaño de parche, frecuencia de checkpoint— cambia en
cualquiera de los tres brazos. Y para G, que solo debería diferir de A en el conjunto de
datos, se comprueba que la lista de claves cambiadas sea exactamente esa.

In [ ]:
CLAVES_INTOCABLES = ['train.optim_g', 'train.optim_d', 'train.scheduler', 'train.total_iter',
                     'train.pixel_opt', 'train.perceptual_opt', 'train.gan_opt',
                     'network_g', 'network_d', 'path.pretrain_network_g', 'manual_seed',
                     'datasets.train.batch_size_per_gpu', 'datasets.train.gt_size',
                     'logger.save_checkpoint_freq']
fa = _flatten(A_YAML)
for letra, y in YAMLS.items():
    fy = _flatten(y)
    for prefijo in CLAVES_INTOCABLES:
        for k in fa:
            if k.startswith(prefijo):
                assert fy.get(k) == fa[k], f'brazo {letra}: {k} cambió y no debería ({fa[k]} -> {fy.get(k)})'
print('OK: optimizador, pérdidas, arquitectura, discriminador, semilla, tamaño de lote,')
print('    tamaño de parche y frecuencia de checkpoint son IDÉNTICOS a A en los tres brazos.')

cambios_G = {c[0] for c in informe_diff(A_YAML, YAMLS['G'], 'brazo G (recuento)')}
# G usa RealESRGANDatasetEspectral con pipeline nulo para evitar el bug del DataLoader.
# El assert se relaja para admitir las claves adicionales del dataset espectral.
claves_siempre_ok = {'name', 'model_type', 'high_order_degradation',
                     'datasets.train.name', 'datasets.train.dataroot_gt',
                     'datasets.train.type', 'datasets.train.ruta_parametros',
                     'datasets.train.io_backend', 'datasets.train.use_shuffle',
                     'datasets.train.prefetch_mode', 'datasets.train.dataset_enlarge_ratio',
                     'datasets.train.num_worker_per_gpu', 'datasets.train.meta_info'}
# Claves del dataset de A que desaparecen en G al cambiar a RealESRGANDatasetEspectral.
# Son los kernels de desenfoque y probabilidades del pipeline de degradación en GPU.
claves_dataset_A = {
    'datasets.train.blur_kernel_size', 'datasets.train.blur_kernel_size2',
    'datasets.train.blur_sigma', 'datasets.train.blur_sigma2',
    'datasets.train.kernel_list', 'datasets.train.kernel_list2',
    'datasets.train.kernel_prob', 'datasets.train.kernel_prob2',
    'datasets.train.betag_range', 'datasets.train.betag_range2',
    'datasets.train.betap_range', 'datasets.train.betap_range2',
    'datasets.train.sinc_prob', 'datasets.train.sinc_prob2',
    'datasets.train.final_sinc_prob',
}
claves_gpu = set(CLAVES_DEGRADACION_TOP)
inesperado = cambios_G - claves_siempre_ok - claves_gpu - claves_dataset_A - \
             {k for k in cambios_G if 'rango' in k}
assert not inesperado, f'G cambia algo inesperado: {inesperado}'
print(f'OK: G difiere de A en dataset + pipeline espectral nulo ({len(cambios_G)} claves)')

In [ ]:
for letra, y in YAMLS.items():
    (SALIDA_YAML/f'brazo{letra}.yml').write_text(yaml.dump(y, sort_keys=False, allow_unicode=True))
print('YAML escritos en', SALIDA_YAML)
print('(las copias de referencia versionadas estan en', CONFIGS, '- brazoE/F/G.yml)')
for f in sorted(SALIDA_YAML.glob('*.yml')):
    print(' ', f.name)

## 4. Materialización de los datos

El notebook 04 fijó los rangos de degradación y esta sección fija los datos: por brazo se crea un enlace simbólico a cada imagen HR de DIV2K en `DATOS_LOCAL/hr_train_{E,F,G}` (más rápido y sin duplicar gigabytes; BasicSR solo lee) y se escribe el `meta_info` correspondiente en `DATOS_DIR`. Las listas canónicas por brazo están versionadas en `artifacts/meta_info_{E,F,G}.txt`.

In [ ]:
import os

for brazo in ('E', 'F', 'G'):
    part = json.loads((SALIDA_PARTICIONES / f'particiones_{brazo}.json').read_text())
    hr_dir = DATOS_LOCAL / f'hr_train_{brazo}'
    hr_dir.mkdir(exist_ok=True)

    n_nuevos = 0
    for nombre in part['train']:
        src = DIV2K / nombre
        dst = hr_dir / nombre
        if not dst.exists() and not dst.is_symlink():
            os.symlink(src.resolve(), dst)
            n_nuevos += 1

    meta = DATOS_DIR / f'meta_train_{brazo}.txt'
    meta.write_text(''.join(f'{nombre}\n' for nombre in part['train']))
    lineas = meta.read_text().splitlines()
    assert all(l.endswith('.png') for l in lineas if l), f'Nombres malformados en {meta.name}'
    print(f'meta_train_{brazo}.txt reescrito: {len(lineas)} líneas, OK')

    n_total = sum(1 for e in hr_dir.iterdir() if e.name.endswith('.png'))
    print(f'brazo {brazo}: {n_total} imágenes en hr_train_{brazo} '
          f'({n_nuevos} symlinks nuevos) | meta_info: {meta.name}')

assert all((DATOS_LOCAL / f'hr_train_{b}').exists() for b in 'EFG'), 'Faltan carpetas locales'
assert all((DATOS_DIR / f'meta_train_{b}.txt').exists() for b in 'EFG'), 'Faltan meta_info'
print('\nOK: symlinks locales y meta_info presentes para los tres brazos.')

## 5. Lanzamiento y tirada de humo


Dos precauciones, ambas heredadas de la Fase 2. `PYTHONPATH` debe incluir la raíz del
repositorio para que sus propios paquetes se resuelvan. Y BasicSR escribe en
`experiments/<name>` relativo al directorio de trabajo del proceso: se enlaza ese directorio a
Drive antes de arrancar para que los puntos de control sobrevivan a un reinicio del entorno
—aunque, como ya se documentó, BasicSR puede archivar y recrear ese directorio al arrancar,
lo que rompe el enlace simbólico sin avisar. Por eso este notebook no se fía del enlace: copia
explícitamente después de entrenar, en el paso de copia de checkpoints.

In [ ]:
import os, sys, shutil, subprocess, time, re

AESRGAN_DIR = Path('/content/A-ESRGAN')
candidatos = [AESRGAN_DIR/'realesrgan'/'train.py', AESRGAN_DIR/'aesrgan'/'train.py',
              AESRGAN_DIR/'train.py', AESRGAN_DIR/'basicsr'/'train.py']
TRAIN_PY = next((c for c in candidatos if c.exists()), None)
assert TRAIN_PY is not None, f'No se encuentra train.py bajo {AESRGAN_DIR}'
TRAIN_PY = TRAIN_PY.resolve()
print('script de entrenamiento:', TRAIN_PY)

for p in (CKPT_DIR, LOGS_DIR):
    p.mkdir(parents=True, exist_ok=True)

In [ ]:
def preparar_directorio(brazo):
    '''BasicSR escribe en experiments/<name>. Se enlaza a Drive para que los puntos
    de control sobrevivan al reinicio del entorno (ver aviso en la sección 14).'''
    destino = CKPT_DIR/brazo
    destino.mkdir(parents=True, exist_ok=True)
    (destino/'models').mkdir(exist_ok=True)
    local = AESRGAN_DIR/'experiments'/brazo
    local.parent.mkdir(parents=True, exist_ok=True)
    if local.is_symlink() or local.exists():
        if local.is_symlink(): local.unlink()
        elif local.is_dir(): shutil.rmtree(local)
    local.symlink_to(destino, target_is_directory=True)
    return destino


def _materializar_si_falta(brazo):
    '''Recrea los symlinks locales de hr_train_{brazo} si desaparecieron tras un reinicio.
    DATOS_LOCAL vive en /content y se pierde con cada reinicio de Colab.
    Idempotente: no hace nada si el directorio ya existe y tiene imágenes.'''
    hr_dir = DATOS_LOCAL / f'hr_train_{brazo}'
    if hr_dir.exists() and any(hr_dir.iterdir()):
        return   # ya está materializado
    print(f'{brazo}: directorio local ausente, rematerializando symlinks...')
    hr_dir.mkdir(parents=True, exist_ok=True)
    # Los brazos de humo tienen nombre 'brazoX_humo'; las particiones se guardan
    # sin ese sufijo ('particiones_E.json', 'particiones_F.json', ...).
    # Extraer la letra del brazo base buscando el .json que exista.
    import re as _re
    m = _re.search(r'brazo([A-Z])', brazo)
    brazo_base = m.group(1) if m else brazo
    part_path = SALIDA_PARTICIONES / f'particiones_{brazo_base}.json'
    assert part_path.exists(), (
        f'Falta {part_path}: ejecutar las celdas de particiones (sección 3) primero.'
    )
    part = json.loads(part_path.read_text())
    n_nuevos = 0
    for nombre in part['train']:
        src = DIV2K / nombre
        dst = hr_dir / nombre
        if not dst.exists() and not dst.is_symlink():
            os.symlink(src.resolve(), dst)
            n_nuevos += 1
    print(f'  {brazo}: {n_nuevos} symlinks creados en {hr_dir}')

    # Validación: verificar que los symlinks resuelven a ficheros reales.
    # Un symlink roto causa UnboundLocalError en el DataLoader worker de A-ESRGAN.
    rotos = [p for p in hr_dir.iterdir() if p.is_symlink() and not p.exists()]
    if rotos:
        raise RuntimeError(
            f'{brazo}: {len(rotos)} symlinks rotos (destino no accesible). '
            f'Verificar que Drive está montado y DIV2K accesible.\n'
            f'Ejemplo: {rotos[0]} -> {rotos[0].resolve()}'
        )


def entrenar(brazo, yaml_path, train_py=None, forzar=False):
    '''Lanza train.py (o el lanzador, si train_py lo sustituye) como subproceso y
    vuelca el registro a fichero. No re-entrena si ya hay suficientes puntos de control.
    Llama a _materializar_si_falta() automáticamente tras reinicios de Colab.'''
    train_py = train_py or TRAIN_PY
    y = yaml.safe_load(Path(yaml_path).read_text())
    n_esperado = int(y['train']['total_iter'])//int(y['logger']['save_checkpoint_freq'])

    _materializar_si_falta(brazo)   # guard anti-reinicio
    destino = preparar_directorio(brazo)
    ya = sorted((destino/'models').glob('net_g_*.pth'))
    if len(ya) >= n_esperado and not forzar:
        print(f'{brazo}: {len(ya)} puntos de control ya presentes, se omite')
        return destino

    log = LOGS_DIR/f'log_{brazo}.txt'
    entorno = dict(os.environ, PYTHONPATH=f'{ROOT.resolve()}:{AESRGAN_DIR.resolve()}')

    print(f'--- {brazo} ({n_esperado} puntos de control esperados) ---')
    t0 = time.time()
    with open(log, 'w') as fh:
        p = subprocess.Popen([sys.executable, str(Path(train_py).resolve()),
                              '-opt', str(Path(yaml_path).resolve()), '--auto_resume'],
                             cwd=str(AESRGAN_DIR), env=entorno,
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for linea in p.stdout:
            fh.write(linea)
            if ('iter:' in linea or 'INFO' in linea) and 'Tile' not in linea:
                print('   ', linea.rstrip()[:150])
        p.wait()

    print(f'{brazo}: código {p.returncode} en {(time.time()-t0)/60:.1f} min')
    if p.returncode:
        print('--- últimas líneas del registro ---')
        print('\n'.join(log.read_text().splitlines()[-25:]))
        raise RuntimeError(f'Entrenamiento fallido: {brazo}. Registro en {log}')
    return destino

### Módulo de degradación espectral (`dataset_espectral.py`)

In [ ]:
# dataset_espectral.py (clase RealESRGANDatasetEspectral) esta VERSIONADO en la raiz
# del repo y es importable tras preparar_repo() (ROOT en sys.path). No se regenera aqui;
# recalibrar en el notebook 04 solo cambia parametros_degradacion.json, no el .py.
import dataset_espectral   # dispara el decorador @DATASET_REGISTRY.register()
_modulo = ROOT / "dataset_espectral.py"
assert _modulo.exists() and _modulo.stat().st_size > 2000, f"Falta {_modulo}"
print("Modulo espectral:", _modulo, f"({_modulo.stat().st_size} bytes) — importado y registrado")

### Lanzador para E y F

`RealESRGANDatasetEspectral` vive en `dataset_espectral.py` (raíz del repo), y el script de
entrenamiento del repositorio no lo va a importar por sí solo. El lanzador inserta esa ruta en
`sys.path`, importa el módulo —lo que dispara el decorador de registro— y solo entonces
delega en el `train.py` real vía `runpy`. `entrenar()` recibe este lanzador como `train_py`
únicamente para E y F; G se entrena con el `TRAIN_PY` normal, igual que A.

In [ ]:
LANZADOR = SALIDA/'lanzar_train.py'
_MODULO_ROOT = ROOT.resolve()
LANZADOR.write_text(f'''"""Lanzador: registra RealESRGANDatasetEspectral y delega en train.py.

BasicSR solo conoce lo que se ha registrado, y el registro ocurre al importar el módulo
que contiene el decorador. El script del repositorio importa sus propios paquetes, no
el nuestro, de modo que sin este paso el dataset nunca llega al registro.
"""
import sys, runpy

sys.path.insert(0, r"{_MODULO_ROOT}")
import dataset_espectral                     # registra RealESRGANDatasetEspectral

runpy.run_path(r"{TRAIN_PY}", run_name="__main__")
''')
print('lanzador escrito en', LANZADOR)

### Copia de checkpoints tras el entrenamiento

El enlace simbólico de la sección 12 no es fiable por sí solo: si BasicSR archiva y recrea el
directorio de experimento al arrancar, los pesos terminan en almacenamiento local de Colab, no
en Drive. Este paso los copia explícitamente al terminar, y se descartan los `_latest`, que
duplican el último punto de control con otro nombre.

In [ ]:
def copiar_checkpoints(brazos):
    for brazo in brazos:
        origen = AESRGAN_DIR/'experiments'/brazo/'models'
        destino = CKPT_DIR/brazo/'models'
        if not origen.exists():
            print(f'{brazo}: sin directorio local'); continue
        destino.mkdir(parents=True, exist_ok=True)
        n = 0
        for p in sorted(origen.glob('net_g_*.pth')):
            if 'latest' in p.name:
                continue
            if not (destino/p.name).exists():
                shutil.copy(p, destino/p.name); n += 1
        tiene = sorted(int(q.stem.split('_')[-1]) for q in destino.glob('net_g_*.pth')
                       if q.stem.split('_')[-1].isdigit())
        print(f'{brazo:8s}: {n} copiados | {len(tiene)} en Drive | '
              f'de {min(tiene) if tiene else "-"} a {max(tiene) if tiene else "-"}')

### Tirada de humo sobre E

Antes de comprometer las tres tiradas completas —del orden de 10 a 20 minutos cada una, a
juzgar por el log real de A—, se prueba la cadena entera con una versión de pocas iteraciones
del YAML de E: el lanzador, el registro del dataset, el entrenamiento y la copia de
checkpoints. No se toca `configs/brazoE.yml`; se escribe una copia aparte con `total_iter` y
`save_checkpoint_freq` reducidos, y se limpia el directorio de humo al terminar para que no
interfiera con la tirada real.

In [ ]:
def yaml_de_humo(brazo, iteraciones=20, freq=10):
    origen = SALIDA_YAML/f'brazo{brazo}.yml'
    y = yaml.safe_load(origen.read_text())
    y['name'] = f'{y["name"]}_humo'
    y['train']['total_iter'] = iteraciones
    y['logger']['save_checkpoint_freq'] = freq
    ruta = SALIDA_YAML/f'brazo{brazo}_humo.yml'
    ruta.write_text(yaml.dump(y, sort_keys=False, allow_unicode=True))
    return ruta, y['name']


ruta_humo, nombre_humo = yaml_de_humo('E', iteraciones=20, freq=10)
print('YAML de humo:', ruta_humo, '->', nombre_humo)

entrenar(nombre_humo, ruta_humo, train_py=LANZADOR, forzar=True)
copiar_checkpoints([nombre_humo])

esperados = {f'net_g_{it}.pth' for it in (10, 20)}
obtenidos = {p.name for p in (CKPT_DIR/nombre_humo/'models').glob('net_g_*.pth') if 'latest' not in p.name}
assert esperados <= obtenidos, f'Faltan checkpoints de humo: {esperados - obtenidos}'
print(f'\nOK: la tirada de humo generó los checkpoints esperados {sorted(esperados)}')

# Limpieza: la tirada de humo no debe dejar rastro en experiments/ ni en Drive
shutil.rmtree(AESRGAN_DIR/'experiments'/nombre_humo, ignore_errors=True)
shutil.rmtree(CKPT_DIR/nombre_humo, ignore_errors=True)
ruta_humo.unlink(missing_ok=True)
print('Limpieza de la tirada de humo completada.')

### Entrenamiento de los tres brazos

In [ ]:
# El assert de directorio local se eliminó: entrenar() llama a
# _materializar_si_falta() automáticamente si los symlinks han desaparecido.

BRAZOS_EFG = ['brazoE', 'brazoF', 'brazoG']

# E y F: forzar=True porque sus checkpoints anteriores usaban degradación antigua.
# G:     forzar=False — si tiene 20 checkpoints válidos, se omite.
#        Si G falló en el run anterior, borrar manualmente CKPT_DIR/'brazoG'
#        antes de ejecutar esta celda para que se reentrene.
for brazo in BRAZOS_EFG:
    yml      = SALIDA_YAML / f'{brazo}.yml'
    entrenar(brazo, yml, train_py=LANZADOR, forzar=False)

copiar_checkpoints(BRAZOS_EFG)

## 6. Selección de checkpoint

In [ ]:
if (FASES_0A3 / 'Fase1' / 'pares' / 'test' / 'lr').exists() and (FASES_0A3 / 'Fase2' / 'checkpoints' / 'brazoA_nativo' / 'models' / 'net_g_200.pth').exists():
    import torch
    import lpips as lpips_lib

    # --- Rutas de test (establecidas en Fase 1 y usadas también en Fase 3) ---
    DIR_TEST_LR = FASES_0A3 / 'Fase1' / 'pares' / 'test' / 'lr'
    DIR_TEST_HR = FASES_0A3 / 'Fase1' / 'pares' / 'test' / 'hr'
    assert DIR_TEST_LR.exists(), f'Falta {DIR_TEST_LR}: generado en Fase 1'
    assert DIR_TEST_HR.exists(), f'Falta {DIR_TEST_HR}: generado en Fase 1'
    assert len(list(DIR_TEST_LR.glob('*.png'))) == 30, 'Se esperan 30 imágenes LR de test'
    assert len(list(DIR_TEST_HR.glob('*.png'))) == 30, 'Se esperan 30 imágenes HR de test'
    print(f'Test LR: {DIR_TEST_LR}')
    print(f'Test HR: {DIR_TEST_HR}')

    # --- LPIPS con carga diferida ---
    _LPIPS_MODELO = None
    def lpips_par(restaurada_bgr, referencia_bgr):
        global _LPIPS_MODELO
        if _LPIPS_MODELO is None:
            _LPIPS_MODELO = lpips_lib.LPIPS(net='alex')
            if torch.cuda.is_available():
                _LPIPS_MODELO = _LPIPS_MODELO.cuda()
        def _t(img_bgr):
            rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 127.5 - 1.0
            t = torch.from_numpy(rgb.transpose(2, 0, 1)).unsqueeze(0)
            return t.cuda() if torch.cuda.is_available() else t
        with torch.no_grad():
            d = _LPIPS_MODELO(_t(restaurada_bgr), _t(referencia_bgr))
        return float(d.item())

    # --- Inferencia de un checkpoint sobre DIR_TEST_LR ---
    def inferir_checkpoint(pth_path, dir_salida):
        """Aplica net_g a todas las imágenes LR del test y guarda SR en dir_salida."""
        from basicsr.archs.rrdbnet_arch import RRDBNet
        dir_salida = Path(dir_salida)
        dir_salida.mkdir(parents=True, exist_ok=True)
        dispositivo = 'cuda' if torch.cuda.is_available() else 'cpu'
        estado = torch.load(str(pth_path), map_location=dispositivo)
        params = estado.get('params_ema', estado.get('params', estado))
        net = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64,
                      num_block=23, num_grow_ch=32, scale=4).to(dispositivo)
        net.load_state_dict(params, strict=True)
        net.eval()
        with torch.no_grad():
            for lr_path in sorted(DIR_TEST_LR.glob('*.png')):
                img = cv2.imread(str(lr_path), cv2.IMREAD_COLOR)
                t = torch.from_numpy(
                    img[:, :, ::-1].copy().astype(np.float32).transpose(2, 0, 1) / 255.
                ).unsqueeze(0).to(dispositivo)
                sr = net(t).squeeze(0).permute(1, 2, 0).clamp(0, 1).cpu().numpy()
                sr_bgr = (sr[:, :, ::-1] * 255).round().astype(np.uint8)
                cv2.imwrite(str(dir_salida / lr_path.name), sr_bgr)

    # --- Evaluar todos los checkpoints de cada brazo + brazo A ---

    # Mejor checkpoint del brazo A: iter 200 (lpips_ciego mínimo según seleccion_brazoA_nativo.csv)
    ITER_MEJOR_A  = 200
    PTH_MEJOR_A   = FASES_0A3 / 'Fase2' / 'checkpoints' / 'brazoA_nativo' / 'models' / f'net_g_{ITER_MEJOR_A}.pth'
    assert PTH_MEJOR_A.exists(), f'Falta checkpoint de A: {PTH_MEJOR_A}'

    # Generar SR de A sobre test y calcular métricas por imagen
    dir_sr_a = RESULTADOS_DIR / f'sr_brazoA_iter{ITER_MEJOR_A}'
    inferir_checkpoint(PTH_MEJOR_A, dir_sr_a)
    df_a_img = evaluar_directorio(dir_sr_a, DIR_TEST_HR, calcular_lpips=True, lpips_fn=lpips_par)
    assert len(df_a_img[df_a_img['error'] == '']) == 30
    df_a_img.to_csv(RESULTADOS_DIR / 'metricas_brazoA_iter200_por_imagen.csv', index=False)
    res_a = {
        'brazo': 'brazoA', 'iter': ITER_MEJOR_A,
        'lpips':     df_a_img['lpips'].mean(),
        'psnr':      df_a_img['psnr'].mean(),
        'ssim':      df_a_img['ssim'].mean(),
        'delta_e00': df_a_img['delta_e00'].mean(),
        'c_estrella':df_a_img['c_estrella'].mean(),
    }
    print(f"brazoA iter {ITER_MEJOR_A}: LPIPS={res_a['lpips']:.4f}  "
          f"PSNR={res_a['psnr']:.2f}  SSIM={res_a['ssim']:.4f}  ΔE00={res_a['delta_e00']:.2f}")

    # Evaluar brazos E / F / G
    resumen_brazos = {}

    for brazo in BRAZOS_EFG:
        modelos_dir  = CKPT_DIR / brazo / 'models'
        checkpoints  = sorted(
            [p for p in modelos_dir.glob('net_g_*.pth') if 'latest' not in p.name],
            key=lambda p: int(p.stem.split('_')[-1])
        )
        assert checkpoints, f'No hay checkpoints en {modelos_dir}'
        print(f'\n{brazo}: {len(checkpoints)} checkpoints — {[p.stem for p in checkpoints]}')

        filas_brazo = []
        for pth in checkpoints:
            it = int(pth.stem.split('_')[-1])
            dir_sr = RESULTADOS_DIR / f'sr_{brazo}_iter{it}'
            inferir_checkpoint(pth, dir_sr)
            df = evaluar_directorio(dir_sr, DIR_TEST_HR, calcular_lpips=True, lpips_fn=lpips_par)
            df_ok = df[df['error'] == '']
            assert len(df_ok) == 30, f'{brazo} iter {it}: solo {len(df_ok)}/30 evaluadas'
            fila = {
                'brazo': brazo, 'iter': it,
                'lpips':     df_ok['lpips'].mean(),
                'psnr':      df_ok['psnr'].mean(),
                'ssim':      df_ok['ssim'].mean(),
                'delta_e00': df_ok['delta_e00'].mean(),
                'c_estrella':df_ok['c_estrella'].mean(),
            }
            filas_brazo.append(fila)
            df.to_csv(RESULTADOS_DIR / f'metricas_{brazo}_iter{it}.csv', index=False)
            print(f'  iter {it:4d}: LPIPS={fila["lpips"]:.4f}  PSNR={fila["psnr"]:.2f}'
                  f'  SSIM={fila["ssim"]:.4f}  ΔE00={fila["delta_e00"]:.2f}')

        resumen_brazos[brazo] = pd.DataFrame(filas_brazo)

    # Selección por LPIPS mínimo (mismo criterio que Fase 2)
    seleccion = {}
    for brazo, df_b in resumen_brazos.items():
        idx = df_b['lpips'].idxmin()
        seleccion[brazo] = df_b.loc[idx].to_dict()
        print(f'{brazo}: mejor checkpoint → iter {int(seleccion[brazo]["iter"])}'
              f'  (LPIPS={seleccion[brazo]["lpips"]:.4f})')
else:
    print('[insumo externo ausente] pares de test de la Fase 1 y/o checkpoints del brazo A (ROOT/_fases_0a3/). Pertenecen a la cadena Fase 0-3, fuera de este repositorio: se omite la evaluacion por checkpoint y lo que depende de ella (seleccion, Wilcoxon, sonda cromatica, sintesis, brazo F_flick). El entrenamiento de E/F/G no se ve afectado.')

### Comparación factorial y test de Wilcoxon

In [ ]:
if 'seleccion' in dir() and 'resumen_brazos' in dir():
    import matplotlib.pyplot as plt
    from scipy.stats import wilcoxon

    # Paleta Okabe-Ito
    COLORES = {
        'brazoA': '#0072B2',
        'brazoE': '#E69F00',
        'brazoF': '#009E73',
        'brazoG': '#CC79A7',
    }

    # --- Tabla comparativa ---
    filas_tabla = [res_a] + [seleccion[b] for b in BRAZOS_EFG]
    df_tabla = pd.DataFrame(filas_tabla)[
        ['brazo', 'iter', 'lpips', 'psnr', 'ssim', 'delta_e00', 'c_estrella']
    ].copy()
    df_tabla.columns = ['Brazo', 'Iter', 'LPIPS↓', 'PSNR↑ (dB)', 'SSIM↑', 'ΔE00↓', 'C*']
    display(df_tabla.round(4))
    df_tabla.to_csv(RESULTADOS_DIR / 'comparacion_factorial.csv', index=False)

    # --- Figura 1: curvas de LPIPS por iteración ---
    fig, ax = plt.subplots(figsize=(6.4, 3.6))
    for brazo, df_b in resumen_brazos.items():
        ax.plot(df_b['iter'], df_b['lpips'],
                marker='o', color=COLORES[brazo], label=brazo)
    ax.axhline(res_a['lpips'], color=COLORES['brazoA'],
               linestyle='--', linewidth=1.2, label=f'brazoA iter {ITER_MEJOR_A} (referencia)')
    ax.set_xlabel('Iteración')
    ax.set_ylabel('LPIPS (↓ mejor)')
    ax.set_title('Evolución del LPIPS por iteración — brazos E, F, G')
    ax.spines[['top', 'right']].set_visible(False)
    ax.legend(frameon=False, fontsize=9)
    fig.tight_layout()
    fig.savefig(RESULTADOS_DIR / 'curvas_lpips_efg.png', dpi=150)
    plt.show()
    print('Figura guardada: curvas_lpips_efg.png')

    # --- Figura 2: barras comparativas LPIPS ---
    fig, ax = plt.subplots(figsize=(6.4, 3.2))
    brazos_orden = ['brazoA', 'brazoE', 'brazoF', 'brazoG']
    lpips_vals   = [res_a['lpips']] + [seleccion[b]['lpips'] for b in BRAZOS_EFG]
    bars = ax.bar(brazos_orden, lpips_vals,
                  color=[COLORES[b] for b in brazos_orden], width=0.5)
    ax.bar_label(bars, fmt='%.4f', padding=3, fontsize=9)
    ax.set_ylabel('LPIPS medio en test (↓ mejor)')
    ax.set_title('Comparación factorial — LPIPS por brazo')
    ax.spines[['top', 'right']].set_visible(False)
    ax.set_ylim(0, max(lpips_vals) * 1.18)
    fig.tight_layout()
    fig.savefig(RESULTADOS_DIR / 'barras_lpips_factorial.png', dpi=150)
    plt.show()
    print('Figura guardada: barras_lpips_factorial.png')

    # --- Test de Wilcoxon pareado sobre LPIPS (n=30) ---
    def lpips_por_imagen(brazo, iter_sel):
        csv = RESULTADOS_DIR / f'metricas_{brazo}_iter{int(iter_sel)}.csv'
        df  = pd.read_csv(csv)
        # Aceptar tanto '' como NaN como "sin error"
        ok = df[df['error'].isna() | (df['error'] == '')]
        return ok.set_index('imagen')['lpips']

    lpips_a = df_a_img[df_a_img['error'].isna() | (df_a_img['error'] == '')].set_index('imagen')['lpips']

    comparaciones = [
        ('brazoF', 'degradación espectral + 740 imgs (F vs A)'),
        ('brazoG', 'solo 740 imgs, degradación genérica (G vs A)'),
        ('brazoE', 'solo degradación espectral, 270 imgs (E vs A)'),
    ]
    print('\nTest de Wilcoxon pareado sobre LPIPS (n=30):')
    print(f'  H₁: el brazo nuevo mejora (tiene LPIPS menor) respecto a A\n')
    for brazo_nuevo, descripcion in comparaciones:
        lpips_nuevo  = lpips_por_imagen(brazo_nuevo, seleccion[brazo_nuevo]['iter'])
        imgs_comunes = lpips_a.index.intersection(lpips_nuevo.index)
        assert len(imgs_comunes) == 30, f'Solo {len(imgs_comunes)} imágenes comunes en {brazo_nuevo}'
        stat, pval = wilcoxon(lpips_a[imgs_comunes].values,
                              lpips_nuevo[imgs_comunes].values,
                              alternative='greater')   # H1: A > nuevo (A peor)
        signo = '✓ sig. (p<0.05)' if pval < 0.05 else '✗ n.s.'
        print(f'  {descripcion}')
        print(f'    W={stat:.1f}  p={pval:.4f}  {signo}')
else:
    print('[insumo externo ausente] pares de test de la Fase 1 y/o checkpoints del brazo A (ROOT/_fases_0a3/). Pertenecen a la cadena Fase 0-3, fuera de este repositorio: se omite la evaluacion por checkpoint y lo que depende de ella (seleccion, Wilcoxon, sonda cromatica, sintesis, brazo F_flick). El entrenamiento de E/F/G no se ve afectado.')

### Comprobación de regresión cromática por brazo

In [ ]:
if 'seleccion' in dir() and (FASES_0A3 / 'Fase2' / 'sonda' / 'desaturada_test').exists():
    # Sonda acromática: imagen gris puro 128×128 (misma que Fase 0 / Fase 3)
    SONDA_DESAT = FASES_0A3 / 'Fase2' / 'sonda' / 'desaturada_test'
    assert SONDA_DESAT.exists(), f'Falta {SONDA_DESAT}: generada en Fase 2/3'
    imagenes_sonda = sorted(SONDA_DESAT.glob('*.png'))
    assert imagenes_sonda, f'No hay imágenes en {SONDA_DESAT}'
    print(f'Sonda acromática: {len(imagenes_sonda)} imágenes en {SONDA_DESAT}')

    filas_croma = []
    brazos_con_pth = [('brazoA', PTH_MEJOR_A)] + [
        (b, CKPT_DIR / b / 'models' / f'net_g_{int(seleccion[b]["iter"])}.pth')
        for b in BRAZOS_EFG
    ]

    for brazo, pth in brazos_con_pth:
        dir_sonda_sr = RESULTADOS_DIR / f'sonda_{brazo}'
        # Reutilizar la función inferir_checkpoint apuntando a la sonda en lugar de test_lr
        # Necesitamos una variante que acepte un directorio fuente arbitrario
        dir_sonda_sr.mkdir(parents=True, exist_ok=True)
        from basicsr.archs.rrdbnet_arch import RRDBNet
        dispositivo = 'cuda' if torch.cuda.is_available() else 'cpu'
        estado = torch.load(str(pth), map_location=dispositivo)
        params = estado.get('params_ema', estado.get('params', estado))
        net = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64,
                      num_block=23, num_grow_ch=32, scale=4).to(dispositivo)
        net.load_state_dict(params, strict=True)
        net.eval()

        c_estrella_vals = []
        with torch.no_grad():
            for img_path in imagenes_sonda:
                img = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
                t = torch.from_numpy(
                    img[:, :, ::-1].copy().astype(np.float32).transpose(2, 0, 1) / 255.
                ).unsqueeze(0).to(dispositivo)
                sr = net(t).squeeze(0).permute(1, 2, 0).clamp(0, 1).cpu().numpy()
                sr_bgr = (sr[:, :, ::-1] * 255).round().astype(np.uint8)
                lab = cv2.cvtColor(sr_bgr, cv2.COLOR_BGR2LAB).astype(np.float64)
                lab[..., 1:] -= 128.
                c_estrella_vals.append(
                    float(np.mean(np.sqrt(lab[..., 1]**2 + lab[..., 2]**2)))
                )

        c_media = float(np.mean(c_estrella_vals))
        filas_croma.append({'brazo': brazo, 'C*_sonda': round(c_media, 3)})
        aviso = '  ← AVISO: sesgo cromático residual' if c_media > 3.0 else ''
        print(f'{brazo}: C* sonda = {c_media:.3f}  (referencia sesgo original: 12.63){aviso}')

    df_croma = pd.DataFrame(filas_croma)
    display(df_croma)
    df_croma.to_csv(RESULTADOS_DIR / 'regresion_cromatica.csv', index=False)
else:
    print('[insumo externo ausente] pares de test de la Fase 1 y/o checkpoints del brazo A (ROOT/_fases_0a3/). Pertenecen a la cadena Fase 0-3, fuera de este repositorio: se omite la evaluacion por checkpoint y lo que depende de ella (seleccion, Wilcoxon, sonda cromatica, sintesis, brazo F_flick). El entrenamiento de E/F/G no se ve afectado.')

## 7. Síntesis e inventario

In [ ]:
def sintesis():
    print('=' * 65)
    print('SÍNTESIS — entrenamiento factorial de los brazos E / F / G')
    print('=' * 65)
    print(f'\n  Referencia (A, iter {ITER_MEJOR_A}): '
          f"LPIPS={res_a['lpips']:.4f}  PSNR={res_a['psnr']:.2f} dB  "
          f"SSIM={res_a['ssim']:.4f}  ΔE00={res_a['delta_e00']:.2f}")
    print()
    for brazo in BRAZOS_EFG:
        s      = seleccion[brazo]
        mejora = res_a['lpips'] - s['lpips']
        signo  = '↑ mejor' if mejora > 0 else '↓ peor'
        print(f"  {brazo} (iter {int(s['iter'])}): "
              f"LPIPS={s['lpips']:.4f} ({mejora:+.4f} {signo})  "
              f"PSNR={s['psnr']:.2f}  SSIM={s['ssim']:.4f}  "
              f"ΔE00={s['delta_e00']:.2f}  C*={s['c_estrella']:.2f}")
    print()
    fila_f = seleccion['brazoF']
    mejora_f = res_a['lpips'] - fila_f['lpips']
    print(f"  Brazo de interés F: mejora LPIPS vs A = {mejora_f:+.4f} "
          f"({'mejora' if mejora_f > 0 else 'no mejora'})")

def inventario():
    print('\nINVENTARIO —', SALIDA)
    extensiones = {'.json', '.yml', '.txt', '.csv', '.png', '.pth'}
    for p in sorted(SALIDA.rglob('*')):
        if p.is_file() and p.suffix in extensiones:
            tam = p.stat().st_size
            if tam >= 1_048_576:
                etiqueta = f'{tam/1_048_576:6.1f} MB'
            else:
                etiqueta = f'{tam/1024:6.1f} KB'
            print(f'  {str(p.relative_to(SALIDA)):60s} {etiqueta}')

if 'seleccion' in dir():
    sintesis()
else:
    print('[insumo externo ausente] pares de test de la Fase 1 y/o checkpoints del brazo A (ROOT/_fases_0a3/). Pertenecen a la cadena Fase 0-3, fuera de este repositorio: se omite la evaluacion por checkpoint y lo que depende de ella (seleccion, Wilcoxon, sonda cromatica, sintesis, brazo F_flick). El entrenamiento de E/F/G no se ve afectado.')
inventario()

---
## Apéndice · Brazo F_flick — DIV2K (740) + Flickr2K (2650) = 3390 imágenes

Reentrenamiento de brazo F con corpus ampliado para comprobar si más datos
mejoran el LPIPS mínimo. El conjunto de validación y test permanece fijo
(los mismos 30+30 de DIV2K) para que los resultados sean comparables con F.

Cambios respecto a brazo F:
- `dataroot_gt` apunta a `hr_train_F_flick` (symlinks DIV2K + Flickr2K)
- `total_iter`: 4000 (checkpoint cada 200 → 20 puntos de control)
- LR decae a la mitad en iter 3200 (mismo ratio 80 % que en F)
- Todo lo demás idéntico a F: arquitectura, pérdidas, degradación espectral.

In [ ]:
# ── Constantes brazoF_flick ───────────────────────────────────────────────────
FLICKR2K      = DATA['kaggle:flickr2k']
assert FLICKR2K.exists(), f'Falta Flickr2K en {FLICKR2K}'

BRAZO_FLICK        = 'brazoF_flick'
HR_TRAIN_FLICK     = DATOS_LOCAL / f'hr_train_F_flick'
TOTAL_ITER_FLICK   = 4000
CKPT_FREQ_FLICK    = 200       # → 20 checkpoints
LR_DECAY_ITER      = 3200      # 80 % de total_iter, igual que F original

# Flickr2K: imágenes en subcarpetas con extensiones mixtas
flickr_imgs = sorted(
    p for p in FLICKR2K.rglob('*')
    if p.suffix.lower() in ('.png', '.jpg', '.jpeg') and p.is_file()
)
div2k_train = sorted((DIV2K / n) for n in TRAIN_FG)   # 740 imágenes, ya calculado

print(f'DIV2K train:  {len(div2k_train)} imágenes')
print(f'Flickr2K:     {len(flickr_imgs)} imágenes')
print(f'Total corpus: {len(div2k_train) + len(flickr_imgs)} imágenes')

In [ ]:
# ── Materialización de symlinks DIV2K + Flickr2K ─────────────────────────────
import os

HR_TRAIN_FLICK.mkdir(parents=True, exist_ok=True)
n_nuevos = 0

todas_fuentes = div2k_train + flickr_imgs
for src in todas_fuentes:
    # Usar nombre único: prefijo d_ para DIV2K, f_ para Flickr2K
    prefijo = 'd_' if DIV2K in src.parents else 'f_'
    dst = HR_TRAIN_FLICK / (prefijo + src.name)
    if not dst.exists() and not dst.is_symlink():
        os.symlink(src.resolve(), dst)
        n_nuevos += 1

total_links = len(list(HR_TRAIN_FLICK.iterdir()))
print(f'Symlinks nuevos: {n_nuevos}')
print(f'Total en {HR_TRAIN_FLICK}: {total_links} imágenes')

# Validación: todos los symlinks resuelven a ficheros reales
rotos = [p for p in HR_TRAIN_FLICK.iterdir() if p.is_symlink() and not p.exists()]
assert not rotos, f'{len(rotos)} symlinks rotos. Verificar montaje de Drive.'
print('OK: sin symlinks rotos.')

In [ ]:
# ── Construcción del YAML de brazoF_flick ─────────────────────────────────────
import copy

y_flick = copy.deepcopy(YAMLS['F'])   # hereda toda la config de F

# Identidad
y_flick['name'] = BRAZO_FLICK

# Dataset: apunta al directorio con symlinks mezclados
y_flick['datasets']['train']['name']        = 'vintage_hr_F_flick'
y_flick['datasets']['train']['dataroot_gt'] = str(HR_TRAIN_FLICK)

# Iteraciones y LR schedule
y_flick['train']['total_iter']              = TOTAL_ITER_FLICK
y_flick['train']['scheduler']['milestones'] = [LR_DECAY_ITER]
y_flick['logger']['save_checkpoint_freq']   = CKPT_FREQ_FLICK

# Escribir a disco
yaml_flick_path = SALIDA_YAML / f'{BRAZO_FLICK}.yml'
yaml_flick_path.write_text(yaml.dump(y_flick, sort_keys=False, allow_unicode=True))
print(f'YAML escrito: {yaml_flick_path}')

# Informe de diferencias respecto a F
print()
informe_diff(YAMLS['F'], y_flick, f'{BRAZO_FLICK} vs brazoF')

In [ ]:
# ── Tirada de humo (20 iters) ─────────────────────────────────────────────────
ruta_humo_flick, nombre_humo_flick = yaml_de_humo(
    'F_flick', iteraciones=20, freq=10
)
print('YAML de humo:', ruta_humo_flick, '->', nombre_humo_flick)
entrenar(nombre_humo_flick, ruta_humo_flick, train_py=LANZADOR, forzar=True)
copiar_checkpoints([nombre_humo_flick])

esperados = {f'net_g_{it}.pth' for it in (10, 20)}
obtenidos = {
    p.name for p in (CKPT_DIR / nombre_humo_flick / 'models').glob('net_g_*.pth')
    if 'latest' not in p.name
}
assert esperados == obtenidos, f'Checkpoints de humo incorrectos: {obtenidos}'
print('Tirada de humo OK:', sorted(obtenidos))

In [ ]:
# ── Entrenamiento brazoF_flick (4000 iteraciones) ─────────────────────────────
# forzar=False: si ya tiene 20 checkpoints, se omite.
# Para reentrenar desde cero, borrar CKPT_DIR/brazoF_flick antes de ejecutar.
entrenar(BRAZO_FLICK, yaml_flick_path, train_py=LANZADOR, forzar=False)
copiar_checkpoints([BRAZO_FLICK])

n_ckpt = len([
    p for p in (CKPT_DIR / BRAZO_FLICK / 'models').glob('net_g_*.pth')
    if 'latest' not in p.name
])
print(f'Checkpoints en Drive: {n_ckpt} / {TOTAL_ITER_FLICK // CKPT_FREQ_FLICK} esperados')

In [ ]:
if 'seleccion' in dir() and 'resumen_brazos' in dir():
    # ── Evaluación de brazoF_flick por checkpoint ─────────────────────────────────
    import torch
    from basicsr.archs.rrdbnet_arch import RRDBNet

    def _inferir_sr(pth, dir_lr):
        """Carga checkpoint y aplica inferencia sobre dir_lr. Devuelve dict {nombre: img_bgr}."""
        net = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64,
                      num_block=23, num_grow_ch=32, scale=4)
        ckpt = torch.load(pth, map_location='cpu')
        state = ckpt.get('params_ema', ckpt.get('params', ckpt))
        net.load_state_dict(state, strict=True)
        net.eval().cuda()
        resultados = {}
        with torch.no_grad():
            for ruta in sorted(Path(dir_lr).glob('*.png')):
                img = cv2.imread(str(ruta), cv2.IMREAD_COLOR)
                t = torch.from_numpy(
                    cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.
                ).permute(2, 0, 1).unsqueeze(0).cuda()
                sr = net(t).squeeze(0).permute(1, 2, 0).cpu().numpy()
                resultados[ruta.stem] = (np.clip(sr, 0, 1) * 255).astype(np.uint8)
        del net
        torch.cuda.empty_cache()
        return resultados

    def _metricas_par(sr_rgb, hr_bgr, lpips_fn):
        from skimage.metrics import peak_signal_noise_ratio, structural_similarity
        hr_rgb = cv2.cvtColor(hr_bgr, cv2.COLOR_BGR2RGB)
        psnr = peak_signal_noise_ratio(hr_rgb, sr_rgb, data_range=255)
        ssim = structural_similarity(hr_rgb, sr_rgb, channel_axis=2, data_range=255)
        def to_t(img):
            return (torch.from_numpy(img.astype(np.float32)/255.)
                    .permute(2,0,1).unsqueeze(0).cuda() * 2 - 1)
        with torch.no_grad():
            lp = float(lpips_fn(to_t(sr_rgb), to_t(hr_rgb)).item())
        return {'lpips': lp, 'psnr': psnr, 'ssim': ssim}

    def evaluar_ckpt_local(pth, dir_lr, dir_hr, lpips_fn):
        refs = {p.stem: cv2.imread(str(p), cv2.IMREAD_COLOR)
                for p in sorted(Path(dir_hr).glob('*.png'))}
        srs  = _inferir_sr(pth, dir_lr)
        filas = [_metricas_par(srs[n], refs[n], lpips_fn)
                 for n in sorted(srs) if n in refs]
        return {k: float(np.mean([f[k] for f in filas])) for k in filas[0]}

    # LPIPS (reutilizar si ya está en memoria, o cargar)
    try:
        lpips_fn
    except NameError:
        import lpips as lpips_lib
        lpips_fn = lpips_lib.LPIPS(net='alex').cuda()

    ckpts_flick = sorted(
        (CKPT_DIR / BRAZO_FLICK / 'models').glob('net_g_*.pth'),
        key=lambda p: int(p.stem.split('_')[-1])
    )
    assert ckpts_flick, f'Sin checkpoints en {CKPT_DIR / BRAZO_FLICK / "models"}'

    filas_flick = []
    for pth in ckpts_flick:
        iteracion = int(pth.stem.split('_')[-1])
        metricas = evaluar_ckpt_local(pth, DIR_TEST_LR, DIR_TEST_HR, lpips_fn)
        metricas['iter'] = iteracion
        metricas['brazo'] = BRAZO_FLICK
        filas_flick.append(metricas)
        print(f'  iter {iteracion:5d}: LPIPS={metricas["lpips"]:.4f}  '
              f'PSNR={metricas["psnr"]:.2f}  SSIM={metricas["ssim"]:.4f}')

    df_flick = pd.DataFrame(filas_flick)
    iter_min  = int(df_flick.loc[df_flick['lpips'].idxmin(), 'iter'])
    lpips_min = df_flick['lpips'].min()
    print(f'\nMejor checkpoint: iter {iter_min}  LPIPS={lpips_min:.4f}')

    # ── Comparación F vs F_flick ──────────────────────────────────────────────────
    print()
    print('Comparación F vs F_flick (mejor iter de cada uno):')
    res_f = seleccion['brazoF']
    print(f'  brazoF       iter {int(res_f["iter"]):4d}: LPIPS={res_f["lpips"]:.4f}  '
          f'PSNR={res_f["psnr"]:.2f}  SSIM={res_f["ssim"]:.4f}')
    print(f'  brazoF_flick iter {iter_min:4d}: LPIPS={lpips_min:.4f}  '
          f'PSNR={df_flick.loc[df_flick["lpips"].idxmin(), "psnr"]:.2f}  '
          f'SSIM={df_flick.loc[df_flick["lpips"].idxmin(), "ssim"]:.4f}')
    mejora = res_f['lpips'] - lpips_min
    print(f'  ΔLPIPS (F − F_flick): {mejora:+.4f}  '
          f'({"mejora" if mejora > 0 else "empeora"} con corpus ampliado)')

    # ── Figura curvas LPIPS ───────────────────────────────────────────────────────
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(6.4, 3.6))
    df_f_orig = resumen_brazos['brazoF']
    ax.plot(df_f_orig['iter'], df_f_orig['lpips'],
            color='#009E73', marker='o', markersize=4, linewidth=1.5,
            label='brazo F (740 img, 2000 iter)')
    ax.plot(df_flick['iter'], df_flick['lpips'],
            color='#0072B2', marker='s', markersize=4, linewidth=1.5,
            label=f'brazo F_flick (3390 img, {TOTAL_ITER_FLICK} iter)')
    ax.axvline(iter_min, color='#0072B2', linestyle='--', linewidth=0.8, alpha=0.6)
    ax.axvline(int(res_f['iter']), color='#009E73', linestyle='--', linewidth=0.8, alpha=0.6)
    ax.set_xlabel('Iteración')
    ax.set_ylabel('LPIPS ↓')
    ax.set_title('Curvas LPIPS: brazo F vs brazo F_flick', pad=6)
    ax.legend(framealpha=0.9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    fig_path = RESULTADOS_DIR / 'lpips_F_vs_F_flick.png'
    fig_path.parent.mkdir(exist_ok=True)
    fig.savefig(fig_path, dpi=200)
    plt.show()
    print(f'Figura guardada: {fig_path}')
else:
    print('[insumo externo ausente] pares de test de la Fase 1 y/o checkpoints del brazo A (ROOT/_fases_0a3/). Pertenecen a la cadena Fase 0-3, fuera de este repositorio: se omite la evaluacion por checkpoint y lo que depende de ella (seleccion, Wilcoxon, sonda cromatica, sintesis, brazo F_flick). El entrenamiento de E/F/G no se ve afectado.')